In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Iceberg v3 deletion vectors on Managed Spark Runtime 4.0 and Google Cloud Lakehouse

Companion notebook for the Apache Spark NYC Meetup talk *Supercharging Apache Spark Lakehouses: Inside Iceberg v3, 1.12 & Road to v4*.

**What this notebook shows**

1. **Iceberg v2:** each merge-on-read `MERGE` commit adds another Parquet position delete file.
2. **In-place upgrade:** setting `format-version` to `3` rewrites no data or delete files.
3. **Iceberg v3:** repeated `MERGE` commits keep exactly one deletion vector per data file.
4. **Multi-engine:** BigQuery queries the same table through the Lakehouse runtime catalog and returns the same results as Spark.

**Architecture:** Managed Spark (Iceberg REST client) → Lakehouse runtime catalog → Cloud Storage (Parquet + Puffin files) ← BigQuery

The notebook first creates the lakehouse itself: a Cloud Storage bucket and a catalog in the Lakehouse runtime catalog. One property, `dataproc.lakehouse.defaultCatalog`, then makes that catalog Spark's default catalog, so this notebook sets no `spark.sql.catalog.*` properties. All table operations are standard Apache Spark SQL on Apache Iceberg.

### Prerequisites

- Enable the APIs: `gcloud services enable dataproc.googleapis.com dataproc-rm.googleapis.com biglake.googleapis.com bigquery.googleapis.com`
- Authenticate where the notebook runs: the Spark and BigQuery clients use Application Default Credentials (`gcloud auth application-default login`), and step 2 uses the gcloud CLI (`gcloud auth login`).
- Runtime 3.0+ sessions run with your end-user credentials, so your account needs:
  - `roles/dataproc.serverlessEditor` to create the interactive session.
  - `roles/storage.admin` to create the bucket in step 2.
  - `roles/biglake.editor` and `roles/storage.objectUser` to create the catalog in step 2 and write Iceberg tables in it.
  - `roles/bigquery.jobUser` to run the BigQuery query in step 7.
- If runtime 4.0 isn't available in your project yet, choose `3.0` in step 1. The notebook sets the same catalog property on both versions.

**Notes:** Iceberg V3 tables in the Lakehouse runtime catalog are in Preview. BigQuery can read them but not write them. Row lineage (`_row_id`) isn't supported there yet, so this notebook doesn't demo it.

In [ ]:
%pip install google-cloud-spark-connect[interactive,spark] google-cloud-bigquery cffi

# Setup: parameters and client library
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
RUNTIME_VERSION = "4.0" # Spark 4.2
SESSION_ID = "iceberg-v3-demo"
LAKEHOUSE_CATALOG = "iceberg-v3-lakehouse"
NAMESPACE = "nyc_meetup"
TABLE_NAME = "events"

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError("Set PROJECT_ID before running this cell.")

TABLE = f"{NAMESPACE}.{TABLE_NAME}"  # Resolves in Spark's default catalog (the Lakehouse catalog).
BUCKET = f"{PROJECT_ID}-{LAKEHOUSE_CATALOG}"  # Cloud Storage bucket for the catalog.
BQ_TABLE = f"{PROJECT_ID}.{LAKEHOUSE_CATALOG}.{NAMESPACE}.{TABLE_NAME}"  # BigQuery P.C.N.T. name.

print(f"Spark table:    {TABLE}")
print(f"BigQuery table: {BQ_TABLE}")

## Create the lakehouse: a bucket, a catalog, and Private Google Access

A lakehouse on Google Cloud starts with two resources: a Cloud Storage bucket that holds the Iceberg data and metadata files, and a catalog in the Lakehouse runtime catalog that tracks the tables in that bucket. Managed Spark Serverless also requires Private Google Access on the region's subnet. The next cell configures all three with the gcloud CLI:

- **Subnet:** enables Private Google Access (`--enable-private-ip-google-access`) on the `default` subnet in `REGION` so the serverless Spark session can reach Google APIs without external IPs.
- **Bucket:** a single-region bucket in `REGION`. The catalog takes its region from the bucket, and BigQuery runs the query in step 7 in that region.
- **Catalog:** a multiple-bucket catalog (`--catalog-type=lakehouse`) that uses the bucket as its default location. It uses end-user credentials, so the catalog checks Cloud Storage access with your identity, the same identity that the Spark session runs as.

You can rerun the cell: it reuses a bucket or catalog that already exists.

In [ ]:
# Create the lakehouse: enable Private Google Access, create Cloud Storage bucket and Lakehouse runtime catalog
import subprocess
def gcloud(*args: str, check: bool = True) -> subprocess.CompletedProcess:
    """Runs a gcloud command in the demo project and raises its error output if it fails."""
    result = subprocess.run(["gcloud", *args, f"--project={PROJECT_ID}"], capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"gcloud {' '.join(args)} failed:\n{result.stderr}")
    return result


# Subnet: Managed Spark requires Private Google Access on the regional subnet.
gcloud("compute", "networks", "subnets", "update", "default", f"--region={REGION}", "--enable-private-ip-google-access")
print(f"Enabled Private Google Access on default subnet in {REGION}")

# Cloud Storage bucket: holds the Parquet data files, delete files, and Iceberg metadata.
if gcloud("storage", "buckets", "describe", f"gs://{BUCKET}", check=False).returncode == 0:
    print(f"Reusing bucket gs://{BUCKET}")
else:
    gcloud("storage", "buckets", "create", f"gs://{BUCKET}", f"--location={REGION}", "--uniform-bucket-level-access")
    print(f"Created bucket gs://{BUCKET} in {REGION}")

# Catalog in the Lakehouse runtime catalog: the Iceberg REST catalog that Spark and BigQuery share.
if gcloud("biglake", "iceberg", "catalogs", "describe", LAKEHOUSE_CATALOG, check=False).returncode == 0:
    print(f"Reusing catalog {LAKEHOUSE_CATALOG}")
else:
    gcloud(
        "biglake", "iceberg", "catalogs", "create", LAKEHOUSE_CATALOG,
        "--catalog-type=lakehouse",
        f"--default-location=gs://{BUCKET}",
        "--credential-mode=end-user",
    )
    print(f"Created catalog {LAKEHOUSE_CATALOG} with default location gs://{BUCKET}")

## Start a Managed Spark session on the new catalog

The `dataproc.lakehouse.defaultCatalog` property points the session at the catalog from the previous step. Managed Service for Apache Spark registers that catalog as Spark's default catalog, `default_catalog`, and generates its Iceberg properties, so the notebook sets no `spark.sql.catalog.*` properties. Without the property, runtime 4.0 and later uses the regional starter catalog (`standard-lh-catalog-REGION`), which the service creates automatically. Under the hood, `default_catalog` is the upstream `org.apache.iceberg.spark.SparkCatalog` with `type=rest`, pointed at the Lakehouse Apache Iceberg REST catalog endpoint. The next cell prints those properties.

`ManagedSparkSession` connects over Spark Connect, so you don't need a local Spark installation or JVM. The session has a fixed ID, so reruns reconnect to the running session and ignore new settings. To apply changes, terminate the session in step 8 or use a new `SESSION_ID`.

In [ ]:
# Start (or reconnect to) the Managed Spark session
from google.cloud.managed_spark_connect import ManagedSparkSession

builder = (
    ManagedSparkSession.builder
    .projectId(PROJECT_ID)
    .location(REGION)
    # .sessionId(SESSION_ID)
    .runtimeVersion(RUNTIME_VERSION)
    .subnetwork("default")
)
# Make the catalog from Spark's default catalog. Without this property, runtime 4.0+ uses the starter catalog.
builder = builder.config("dataproc.lakehouse.catalog", f"projects/{PROJECT_ID}/catalogs/{LAKEHOUSE_CATALOG}")
spark = builder.getOrCreate()

# Workaround: Dataproc's SessionController overwrites spark.sql.catalog.default_catalog.warehouse with
# the catalog's default_location (gs://...) at session creation time. Set it to the bl:// URI before
# default_catalog is lazily initialized on first SQL execution.
spark.conf.set("spark.sql.catalog.default_catalog.warehouse", f"bl://projects/{PROJECT_ID}/catalogs/{LAKEHOUSE_CATALOG}")

catalog = spark.conf.get("spark.sql.defaultCatalog")
if catalog != "default_catalog":
    raise RuntimeError(f"Expected the Lakehouse default catalog, but Spark's default catalog is {catalog!r}.")

# Everything printed here was configured by the runtime (plus the warehouse workaround above).
print(f"Apache Spark {spark.version}")
for key in ["spark.sql.defaultCatalog"] + [
    f"spark.sql.catalog.{catalog}{suffix}" for suffix in ("", ".type", ".uri", ".warehouse")
]:
    print(f"  {key} = {spark.conf.get(key, '<not set>')}")


def merge_status(row_id: int, status: str) -> None:
    """Runs one small MERGE commit that updates a single row."""
    spark.sql(f"""
        MERGE INTO {TABLE} t
        USING (SELECT {row_id} AS id, '{status}' AS status) s
        ON t.id = s.id
        WHEN MATCHED THEN UPDATE SET t.status = s.status
    """)


def show_delete_files() -> None:
    """Shows the table's live delete files from Iceberg's delete_files metadata table."""
    spark.sql(f"""
        SELECT file_format,
               record_count AS deleted_rows,
               element_at(split(file_path, '/'), -1) AS delete_file,
               element_at(split(referenced_data_file, '/'), -1) AS referenced_data_file,
               content_offset,
               content_size_in_bytes
        FROM {TABLE}.delete_files
    """).show(truncate=False)


spark  # Shows links to the session details page and the Spark UI.

# Iceberg v2 adds a delete file for every commit

With merge-on-read, a `MERGE` doesn't rewrite the data file. Instead, it writes the updated row as new data and records the old row's position in a **position delete file**: Parquet rows of `(file_path, pos)`. With partition-scoped deletes, which is classic v2 behavior, every commit adds another delete file, and readers must open and apply all of them.

This cell sets `write.delete.granularity = 'partition'` to show that classic behavior. Recent Iceberg Spark writers default to file-scoped deletes and can merge earlier deletes on each commit. That's a v2-era workaround that v3 deletion vectors make part of the spec.

In [ ]:
# Iceberg v2, two MERGEs produce two delete files
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NAMESPACE}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE} PURGE")
spark.sql(f"""
    CREATE TABLE {TABLE} (id BIGINT, status STRING)
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read',
        'write.delete.granularity' = 'partition'
    )
""")

# Seed 100,000 rows as a single Parquet data file.
spark.range(1, 100_001).selectExpr("id", "'active' AS status").coalesce(1).writeTo(TABLE).append()

# Two small MERGE commits against that one data file.
merge_status(10, "upgraded")
merge_status(20, "upgraded")

show_delete_files()  # Expect 2 PARQUET position delete files.

## Upgrade to v3 in place

The upgrade changes only table metadata. Existing data files and v2 delete files are untouched and remain valid. You can't downgrade a v3 table to v2.

In [ ]:
# Upgrade the table to Iceberg v3
spark.sql(f"ALTER TABLE {TABLE} SET TBLPROPERTIES ('format-version' = '3')")
spark.sql(f"SHOW TBLPROPERTIES {TABLE} ('format-version')").show()
spark.sql(f"SELECT count(*) AS total_rows, count_if(status = 'upgraded') AS upgraded_rows FROM {TABLE}").show()

## Iceberg v3 keeps one deletion vector per data file

In v3, Spark records deletes as **deletion vectors (DVs)**: Roaring bitmaps of deleted row positions stored as blobs in Puffin files. Each DV is located by `content_offset` and `content_size_in_bytes`. The spec allows at most one DV per data file. When a commit deletes more rows from a data file that already has a DV, the writer merges the old bitmap into a new DV and removes the old DV in the same commit.

To mimic data arriving after the upgrade, the next cell appends a new 100,000-row data file and runs three `MERGE` commits against it. The two v2 delete files from Part 1 stay in place until the next compaction, which also shows that the upgrade rewrote nothing.

In [ ]:
# Iceberg v3, three MERGEs still leave one deletion vector
# New data lands after the upgrade: another 100,000 rows as one data file.
spark.range(100_001, 200_001).selectExpr("id", "'active' AS status").coalesce(1).writeTo(TABLE).append()

# Three MERGE commits against that new data file.
for row_id in (100_100, 100_200, 100_300):
    merge_status(row_id, "v3_merged")

show_delete_files()  # Expect the 2 legacy PARQUET files plus exactly 1 PUFFIN DV with deleted_rows = 3.

# Commit log: total delete files grows with each v2 commit, then stays flat in v3 because each commit replaces the DV.
spark.sql(f"""
    SELECT operation,
           summary['added-delete-files'] AS added_delete_files,
           summary['added-dvs']          AS added_dvs,
           summary['removed-dvs']        AS removed_dvs,
           summary['total-delete-files'] AS total_delete_files
    FROM {TABLE}.snapshots
    ORDER BY committed_at
""").show()

## Query the same table from BigQuery

Spark committed every snapshot to the Lakehouse runtime catalog, so BigQuery can query the table immediately, with no data copy and no external table to register. BigQuery uses the four-part name `project.catalog.namespace.table` (P.C.N.T.). It reads the same Parquet data files, legacy delete files, and Puffin deletion vector as Spark, so both engines should return the same counts.

In [ ]:
# Run the same query in Spark and in BigQuery
from google.cloud import bigquery

query = "SELECT status, COUNT(*) AS row_count FROM {} GROUP BY status ORDER BY status"

print("Spark:")
spark.sql(query.format(TABLE)).show()

print(f"BigQuery (`{BQ_TABLE}`):")
bq = bigquery.Client(project=PROJECT_ID, location=REGION)
for row in bq.query(query.format(f"`{BQ_TABLE}`")).result():
    print(f"  {row.status:<10} {row.row_count:>8,}")

## Cleanup

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE} PURGE")
spark.sql(f"DROP NAMESPACE IF EXISTS {NAMESPACE}")
spark.stop(terminate=True)  # Terminates the named serverless session.
print("Dropped the demo table and namespace, and terminated the session.")

# The catalog and bucket stay in place because they may have reused existing ones. To delete them:
#   gcloud biglake iceberg catalogs delete LAKEHOUSE_CATALOG --project=PROJECT_ID
#   gcloud storage rm --recursive gs://BUCKET